In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

True

In [2]:
api_key = os.getenv("GROQ_API_KEY")

if api_key is None:
    raise ValueError("GROQ_API_KEY is not set in the environment variables.")
else:
    print("GROQ_API_KEY is set.")

GROQ_API_KEY is set.


In [3]:
model = ChatGroq(
    api_key=api_key,
    model="openai/gpt-oss-20b",
    temperature=0.7,
    max_retries=3,
    max_tokens=1024,
    timeout=30
)

In [11]:
response = model.invoke("Write a short poem about the beauty of nature.")

In [13]:
print(response.content)

In dawn’s first blush the meadow sighs,  
A silver mist upon the rise.  
Birdsong stitches sky with thread,  
Leaves hum in the quiet spread.  

Mountains stand, their peaks in grace,  
Echoing the earth’s embrace.  
Rivers carve the ancient stone,  
Carrying stories, softly grown.  

Night unfurls a velvet shawl,  
Stars ignite the darkened hall.  
Nature’s breath, a timeless rhyme,  
A masterpiece in every clime.


In [16]:

system_prompt = """
You are a professional fiction author, creative storyteller, and world-building expert.

Your task is to create an original, engaging, and well-structured fictional book based on the user's instructions.

Follow these guidelines:

1. Originality: Create original characters, locations, events, conflicts, and plotlines. Avoid copying existing books or authors' distinctive writing styles.

2. World Building: Develop a believable fictional world with its own history, culture, environment, rules, and social structures.

3. Characters: Give every major character:
   - A distinct personality.
   - A personal history.
   - Motivations and ambitions.
   - Strengths and weaknesses.
   - Character development throughout the story.

4. Plot Development: Structure the story with:
   - An engaging opening.
   - A central conflict.
   - Rising tension and unexpected developments.
   - Meaningful character decisions.
   - A compelling climax.
   - A satisfying conclusion.

5. Writing Quality: Use vivid descriptions, natural dialogue, emotional depth, and engaging storytelling. Avoid unnecessary repetition, predictable plotlines, and shallow characters.

6. Chapter Structure: Divide the book into clearly numbered chapters. Give each chapter a meaningful title and ensure that events flow naturally from one chapter to another.

7. Consistency: Maintain continuity in character names, relationships, timelines, locations, and events throughout the book.

8. Reader Engagement: Make the story immersive and encourage readers to remain interested in what happens next.

9. Creative Freedom: Where the user's instructions leave room for creativity, make thoughtful creative decisions without repeatedly asking for clarification.

10. Output: Write in well-organized prose using appropriate paragraphs, chapter headings, and dialogue formatting.

Your goal is to produce a professionally written fictional book that feels original, immersive, emotionally engaging, and enjoyable to read.
"""


In [18]:

from pydantic import BaseModel, Field
from typing import List


class Character(BaseModel):
    name: str = Field(description="Name of the character")
    age: int = Field(description="Age of the character")
    personality: str = Field(description="Character's personality")
    background: str = Field(description="Character's personal history")
    motivation: str = Field(description="What drives the character")


class Chapter(BaseModel):
    chapter_number: int = Field(description="Chapter number")
    title: str = Field(description="Chapter title")
    content: str = Field(description="Full content of the chapter")


class Book(BaseModel):
    title: str = Field(description="Title of the fictional book")
    genre: str = Field(description="Book genre, e.g. fantasy, thriller, romance")
    description: str = Field(description="Brief summary of the book")
    characters: List[Character] = Field(description="Major characters in the book")
    chapters: List[Chapter] = Field(description="All chapters of the book")


In [19]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt=system_prompt,
    response_format=Book
)

In [21]:
from langchain.messages import HumanMessage

message = HumanMessage(
    content="""
    Write a fantasy novel titled 'The Kingdom Beneath Lagos'.

    The story should follow a young Nigerian man who discovers
    a hidden kingdom beneath Lagos Island.

    Include:
    - 3 major characters.
    - 5 chapters.
    - Mystery, adventure, betrayal, and unexpected twists.
    - A compelling ending.

    Make the story immersive and engaging.
    """
)


In [22]:
for chunk, metadata in agent.stream(
    {"messages": [message]},
    stream_mode="messages"
):
    if chunk.content:
        print(chunk.content, end="", flush=True)

{"title":"The Kingdom Beneath Lagos","genre":"Fantasy Adventure","description":"When a curious young man named Kelechi stumbles upon a forgotten map in a bustling Lagos market, he is thrust into a subterranean world beneath the city’s bustling streets. Joined by a brilliant scholar, Nneka, and pursued by a greedy businessman, Oke, Kelechi must navigate ancient mysteries, treacherous politics, and betrayals to protect a hidden kingdom that could alter the fate of Lagos itself.","characters":[{"name":"Kelechi Adeyemi","age":22,"personality":"Curious, determined, and quick-witted, yet often impulsive and trusting.","background":"Orphaned at a young age, raised by his aunt in the vibrant streets of Lagos. He has always felt a tug toward the unknown, fueled by stories his aunt told of the city’s hidden depths.","motivation":"To uncover the truth about his lineage and find a place where he truly belongs."},{"name":"Nneka Okafor","age":28,"personality":"Stoic, analytical, and fiercely protect

In [4]:
from langchain.tools import tool
import math

@tool("square_root", description="Calculate the square root of a number.")
def tool1(number: float) -> float:
    """Calculate the square root of a number."""
    return math.sqrt(number)

In [5]:
system_prompt = """
You help with basic mathematical calculations
"""

In [7]:
from langchain.agents import create_agent

agent_with_tools = create_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[tool1]
    )



In [ ]:
response = agent_with_tools.invoke({"messages": [{"role": "user", "content": "What is the square root of 256?"}]})
print(response["messages"][0]["content"])

{'messages': [HumanMessage(content='What is the square root of 256?', additional_kwargs={}, response_metadata={}, id='caee7b23-ff75-418d-b355-8cb44ed74171'), AIMessage(content='', additional_kwargs={'reasoning_content': "We need to compute sqrt(256). It's 16. Use tool.", 'tool_calls': [{'id': 'fc_31446781-bac1-4a6f-865d-e3da03a75e37', 'function': {'arguments': '{"number":256}', 'name': 'square_root'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 140, 'total_tokens': 179, 'completion_time': 0.043731709, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.012913296, 'prompt_tokens_details': None, 'queue_time': 0.994494561, 'total_time': 0.056645005}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_228717f27c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1eef2-182a-7258-a1a2-29a6ed614821-0', tool_calls=[{'name': 'square_root',